# Tweet Meme vs Real Event Classifier

End-to-end pipeline:

1. **Data Collection** — load tweets with timestamps
2. **Preprocessing** — clean text (URLs, mentions, special chars, lowercase)
3. **SBERT Embedding** — `all-mpnet-base-v2`, save `.npy`
4. **BERTopic Clustering** — extract top keywords, peak time, size per cluster
5. **GDELT Querying** — verify clusters against real-world news (±6 h window)
6. **Label Assignment** — `real_event` (1) if GDELT match else `meme` (0)
7. **Feature Engineering** — SBERT (768-d) + velocity features
8. **XGBoost Training** — 80/20 split, classification report + confusion matrix
9. **Inference** — raw tweet → label + confidence

> Each step saves intermediates to disk so cells can be re-run independently.
> Target: Python 3.10.


## 0. Setup — Install & Import Dependencies

Run once per environment. Pinned versions known-compatible with Python 3.10.


In [ ]:
# Uncomment on first run to install dependencies.
# !pip install -q "sentence-transformers>=2.7.0" "bertopic>=0.16.0" \
#     "xgboost>=2.0.0" "scikit-learn>=1.3.0" "pandas>=2.0.0" \
#     "numpy>=1.24.0" "requests>=2.31.0" "umap-learn>=0.5.5" "hdbscan>=0.8.33"

import os
import re
import json
import time
import random
import pickle
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
import requests

ARTIFACTS = Path("artifacts_meme_vs_event")
ARTIFACTS.mkdir(exist_ok=True)

RAW_CSV          = ARTIFACTS / "raw_tweets.csv"
CLEAN_CSV        = ARTIFACTS / "clean_tweets.csv"
EMBEDDINGS_NPY   = ARTIFACTS / "embeddings.npy"
TOPIC_MODEL_PKL  = ARTIFACTS / "bertopic_model.pkl"
CLUSTERS_CSV     = ARTIFACTS / "clusters.csv"
GDELT_CACHE_JSON = ARTIFACTS / "gdelt_cache.json"
LABELED_CSV      = ARTIFACTS / "labeled_tweets.csv"
FEATURES_NPY     = ARTIFACTS / "features.npy"
LABELS_NPY       = ARTIFACTS / "labels.npy"
XGB_MODEL_JSON   = ARTIFACTS / "xgb_model.json"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Artifacts directory: {ARTIFACTS.resolve()}")


## 1. Data Collection

Load a tweet dataset. The notebook accepts any CSV with columns `text` and `timestamp`
(ISO-8601 or Unix seconds). If no input file is provided, it falls back to a small
synthetic sample so the pipeline can run end-to-end.

For real work, point `INPUT_CSV` at your PHEME / CrisisLex / disaster-tweets file, e.g.:

```
INPUT_CSV = "disaster.csv"  # must contain text + timestamp columns
```


In [ ]:
# Point this to your tweet CSV. Must have `text` and `timestamp` columns
# (timestamp can be ISO string or unix seconds). Set to None to use the synthetic sample.
INPUT_CSV = None  # e.g. "disaster.csv"

TEXT_COL = "text"
TIME_COL = "timestamp"


def _load_input(path: str | None) -> pd.DataFrame:
    """Load tweets from CSV or fall back to a synthetic demo set."""
    if path and Path(path).exists():
        df = pd.read_csv(path)
        if TEXT_COL not in df.columns or TIME_COL not in df.columns:
            raise ValueError(
                f"CSV must have '{TEXT_COL}' and '{TIME_COL}' columns; got {df.columns.tolist()}"
            )
        return df[[TEXT_COL, TIME_COL]].copy()

    print("No INPUT_CSV provided — using synthetic demo tweets.")
    base = pd.Timestamp("2024-02-06 04:00", tz="UTC")
    demo = []
    # Cluster A: real event (earthquake)
    for i in range(60):
        demo.append({
            TEXT_COL: random.choice([
                "6.2 magnitude earthquake just hit southern Turkey, buildings shaking",
                "Massive earthquake in Turkey, evacuations underway in Adana",
                "Felt a huge earthquake near the Syrian border, lights flickered",
                "Turkey earthquake: emergency services dispatched to collapse sites",
            ]),
            TIME_COL: base + timedelta(minutes=i * 2),
        })
    # Cluster B: real event (AWS outage)
    for i in range(45):
        demo.append({
            TEXT_COL: random.choice([
                "AWS us-east-1 is down again, our API is returning 500s",
                "Widespread AWS outage affecting Lambda and S3",
                "us-east-1 region reporting elevated error rates on AWS console",
                "AWS outage is taking half the internet down right now",
            ]),
            TIME_COL: base + timedelta(hours=6) + timedelta(minutes=i * 3),
        })
    # Cluster C: meme
    for i in range(50):
        demo.append({
            TEXT_COL: random.choice([
                "skibidi toilet no cap fr fr 💀💀",
                "my sleep schedule is literally skibidi ohio rizz",
                "gyatt level 9000 on this sigma edit",
                "bro really said 'it's giving ohio' 😭",
            ]),
            TIME_COL: base + timedelta(hours=12) + timedelta(minutes=i * 4),
        })
    return pd.DataFrame(demo)


raw_df = _load_input(INPUT_CSV)
raw_df[TIME_COL] = pd.to_datetime(raw_df[TIME_COL], utc=True, errors="coerce")
raw_df = raw_df.dropna(subset=[TEXT_COL, TIME_COL]).reset_index(drop=True)

raw_df.to_csv(RAW_CSV, index=False)
print(f"Loaded {len(raw_df)} tweets -> {RAW_CSV}")
raw_df.head()


## 2. Preprocessing

Normalize the raw text so embeddings and keyword extraction aren't dominated by
noise (URLs, mentions, hashtag symbols, emojis, extra whitespace). We keep a
copy of the original text in `raw_text` because inference should show the user's
original string.


In [ ]:
_URL_RE      = re.compile(r"https?://\S+|www\.\S+")
_MENTION_RE  = re.compile(r"@\w+")
_HASHTAG_RE  = re.compile(r"#")
_NON_WORD_RE = re.compile(r"[^a-z0-9\s]")
_WS_RE       = re.compile(r"\s+")


def clean_tweet(text: str) -> str:
    """Lowercase, drop URLs/mentions, strip hashtag symbol, remove special chars."""
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = _URL_RE.sub(" ", t)
    t = _MENTION_RE.sub(" ", t)
    t = _HASHTAG_RE.sub(" ", t)      # keep the word, drop the '#'
    t = _NON_WORD_RE.sub(" ", t)     # drop emojis/punctuation
    t = _WS_RE.sub(" ", t).strip()
    return t


clean_df = raw_df.rename(columns={TEXT_COL: "raw_text"}).copy()
clean_df["clean_text"] = clean_df["raw_text"].map(clean_tweet)

# Drop rows that became empty after cleaning (pure URLs / emojis).
clean_df = clean_df[clean_df["clean_text"].str.len() > 0].reset_index(drop=True)

clean_df.to_csv(CLEAN_CSV, index=False)
print(f"Kept {len(clean_df)} tweets after cleaning -> {CLEAN_CSV}")
clean_df[["raw_text", "clean_text"]].head()


## 3. SBERT Embedding

Encode each cleaned tweet with `all-mpnet-base-v2` (768-dim). Embeddings are
cached to disk so downstream steps (BERTopic, XGBoost) can be re-run without
re-encoding.


In [ ]:
from sentence_transformers import SentenceTransformer

SBERT_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

# Load once and keep in memory for later (inference cell re-uses this).
sbert = SentenceTransformer(SBERT_MODEL_NAME)

# Re-use cached embeddings if available AND they match the row count.
if EMBEDDINGS_NPY.exists():
    cached = np.load(EMBEDDINGS_NPY)
    if cached.shape[0] == len(clean_df):
        embeddings = cached
        print(f"Loaded cached embeddings: {embeddings.shape}")
    else:
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    embeddings = sbert.encode(
        clean_df["clean_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # cosine-ready
    )
    np.save(EMBEDDINGS_NPY, embeddings)
    print(f"Saved embeddings {embeddings.shape} -> {EMBEDDINGS_NPY}")


## 4. BERTopic Clustering

Run BERTopic on the pre-computed embeddings. We skip BERTopic's internal
embedding step by passing `embedding_model=None` and our own `embeddings=`.

For each cluster (topic id ≥ 0; -1 = outlier/noise) we extract:

- top 5 keywords (via BERTopic's c-TF-IDF)
- peak timestamp (median tweet time inside the cluster)
- tweet count


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Small-corpus-friendly config: drop the min_df floor and let HDBSCAN form tiny clusters.
vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)

topic_model = BERTopic(
    embedding_model=None,          # we pass our own embeddings
    vectorizer_model=vectorizer,
    min_topic_size=max(5, len(clean_df) // 50),
    calculate_probabilities=False,
    verbose=False,
)

topics, _ = topic_model.fit_transform(clean_df["clean_text"].tolist(), embeddings)
clean_df["topic"] = topics

# Persist the fitted topic model for re-use (e.g. inference on new tweets).
with open(TOPIC_MODEL_PKL, "wb") as fh:
    pickle.dump(topic_model, fh)


def _cluster_summary(df: pd.DataFrame, model: BERTopic) -> pd.DataFrame:
    """One row per cluster with keywords, peak time, and size."""
    rows = []
    for topic_id in sorted(t for t in df["topic"].unique() if t != -1):
        sub = df[df["topic"] == topic_id]
        keywords = [w for w, _ in model.get_topic(topic_id)[:5]]
        rows.append({
            "topic_id": int(topic_id),
            "size": int(len(sub)),
            "peak_timestamp": sub[TIME_COL].median(),
            "keywords": keywords,
        })
    return pd.DataFrame(rows).sort_values("size", ascending=False).reset_index(drop=True)


clusters_df = _cluster_summary(clean_df, topic_model)
# Serialize keywords as JSON so the CSV round-trips cleanly.
clusters_df.assign(keywords=clusters_df["keywords"].map(json.dumps)).to_csv(CLUSTERS_CSV, index=False)

print(f"Found {len(clusters_df)} clusters (excluding noise).")
clusters_df


## 5. GDELT Querying

For every cluster we hit the GDELT DOC 2.0 API with the cluster's top keywords
and a ±6 h window around its peak timestamp. If GDELT returns at least one
article, the cluster is considered a real-world event.

- Uses exponential backoff for rate limits / 5xx errors
- Caches results to `gdelt_cache.json` so cells can be re-run cheaply
- Returns `{topic_id: {"match": bool, "article_count": int}}`


In [ ]:
GDELT_URL = "https://api.gdeltproject.org/api/v2/doc/doc"
GDELT_WINDOW_HOURS = 6
GDELT_MIN_ARTICLES = 1           # threshold for "event confirmed"
GDELT_MAX_RETRIES  = 5
GDELT_BASE_DELAY   = 1.5         # seconds


def _gdelt_ts(ts: pd.Timestamp) -> str:
    """GDELT expects timestamps in YYYYMMDDHHMMSS UTC."""
    return ts.tz_convert("UTC").strftime("%Y%m%d%H%M%S")


def _build_query(keywords: list[str]) -> str:
    """AND the top 2 keywords, OR with the next ones — tight enough to avoid false matches."""
    kws = [k for k in keywords if k][:5]
    if not kws:
        return ""
    if len(kws) == 1:
        return kws[0]
    core = f'"{kws[0]}" "{kws[1]}"'
    extra = " OR ".join(f'"{k}"' for k in kws[2:])
    return f"({core}) OR ({extra})" if extra else core


def query_gdelt(keywords: list[str], peak_ts: pd.Timestamp) -> dict:
    """Query GDELT with exponential backoff. Returns {'match': bool, 'article_count': int}."""
    query = _build_query(keywords)
    if not query or pd.isna(peak_ts):
        return {"match": False, "article_count": 0, "error": "empty_query"}

    start = _gdelt_ts(peak_ts - timedelta(hours=GDELT_WINDOW_HOURS))
    end   = _gdelt_ts(peak_ts + timedelta(hours=GDELT_WINDOW_HOURS))

    params = {
        "query": query,
        "mode": "artlist",
        "format": "json",
        "maxrecords": "25",
        "startdatetime": start,
        "enddatetime": end,
    }

    delay = GDELT_BASE_DELAY
    for attempt in range(1, GDELT_MAX_RETRIES + 1):
        try:
            resp = requests.get(GDELT_URL, params=params, timeout=20)
            if resp.status_code == 200:
                # GDELT occasionally returns empty body on success — treat as zero hits.
                try:
                    data = resp.json() if resp.text.strip() else {}
                except ValueError:
                    data = {}
                articles = data.get("articles", []) if isinstance(data, dict) else []
                count = len(articles)
                return {"match": count >= GDELT_MIN_ARTICLES, "article_count": count}
            # Retryable statuses: 429 rate limit, 5xx server errors.
            if resp.status_code in {429, 500, 502, 503, 504}:
                raise requests.RequestException(f"status {resp.status_code}")
            return {"match": False, "article_count": 0, "error": f"http_{resp.status_code}"}
        except requests.RequestException as exc:
            if attempt == GDELT_MAX_RETRIES:
                return {"match": False, "article_count": 0, "error": str(exc)}
            sleep_for = delay + random.uniform(0, 0.5)  # jitter
            time.sleep(sleep_for)
            delay *= 2  # exponential backoff

    return {"match": False, "article_count": 0, "error": "max_retries"}


cache: dict[str, dict] = {}
if GDELT_CACHE_JSON.exists():
    cache = json.loads(GDELT_CACHE_JSON.read_text())

gdelt_results: dict[int, dict] = {}
for _, row in clusters_df.iterrows():
    tid = int(row["topic_id"])
    cache_key = f"{tid}:{row['peak_timestamp']}:{','.join(row['keywords'])}"
    if cache_key in cache:
        gdelt_results[tid] = cache[cache_key]
        continue
    result = query_gdelt(row["keywords"], row["peak_timestamp"])
    cache[cache_key] = result
    gdelt_results[tid] = result
    time.sleep(0.4)  # be polite between clusters

GDELT_CACHE_JSON.write_text(json.dumps(cache, indent=2, default=str))

print("GDELT results:")
for tid, res in gdelt_results.items():
    print(f"  topic {tid}: {res}")


## 6. Label Assignment

Combine GDELT verification with cluster **lifespan** so we can separate
sustained real-world events from short viral memes that briefly trend.

Rule:

| Condition | Label |
|---|---|
| GDELT match **AND** `lifespan_hours > 12` | `real_event` (1) |
| GDELT match **AND** `lifespan_hours < 6`  | `meme` (0) |
| GDELT match **AND** `6 <= lifespan_hours <= 12` | ambiguous — default to `meme` (0) |
| No GDELT match | `meme` (0) |

Noise cluster (`topic == -1`) is also labeled `meme` — those tweets didn't form
any coherent topic.


In [ ]:
# Lifespan per cluster drives the labeling rule AND feeds feature engineering below.
def _cluster_lifespan_hours(df: pd.DataFrame) -> dict[int, float]:
    """Hours between the first and last tweet of each cluster."""
    spans: dict[int, float] = {}
    for tid, sub in df.groupby("topic"):
        times = sub[TIME_COL]
        spans[int(tid)] = (times.max() - times.min()).total_seconds() / 3600.0
    return spans


lifespan_hours = _cluster_lifespan_hours(clean_df)


def _label_for_cluster(topic_id: int) -> int:
    """Apply GDELT + lifespan rule. Outlier cluster (-1) always -> meme."""
    if topic_id == -1:
        return 0
    gdelt = gdelt_results.get(int(topic_id), {}).get("match", False)
    span = lifespan_hours.get(int(topic_id), 0.0)
    if gdelt and span > 12:
        return 1  # real_event: confirmed by news AND sustained
    if gdelt and span < 6:
        return 0  # meme: briefly viral even if some news matched
    return 0      # no match, or ambiguous 6-12h window


labeled_df = clean_df.copy()
labeled_df["lifespan_hours"] = labeled_df["topic"].map(lifespan_hours).fillna(0.0)
labeled_df["label"] = labeled_df["topic"].map(_label_for_cluster).astype(int)
labeled_df["label_name"] = labeled_df["label"].map({0: "meme", 1: "real_event"})

labeled_df.to_csv(LABELED_CSV, index=False)

print("Per-cluster decisions:")
for tid in sorted(t for t in labeled_df["topic"].unique() if t != -1):
    print(
        f"  topic {tid:>2}  size={int((labeled_df['topic'] == tid).sum()):>4}"
        f"  lifespan={lifespan_hours.get(tid, 0):.2f}h"
        f"  gdelt_match={gdelt_results.get(tid, {}).get('match', False)}"
        f"  -> {labeled_df.loc[labeled_df['topic'] == tid, 'label_name'].iloc[0]}"
    )

print("\nLabel distribution:")
print(labeled_df["label_name"].value_counts())
print(f"\nSaved -> {LABELED_CSV}")
labeled_df.head()


## 7. Feature Engineering

Per-tweet feature vector = **SBERT embedding (768-d)** concatenated with three
**velocity features** describing the tweet's cluster:

- `tweet_rate_per_hour` — `cluster_size / lifespan_hours`
- `lifespan_hours` — `last_tweet_time - first_tweet_time` (in hours)
- `peak_to_decay_ratio` — `tweets_in_first_6h / tweets_in_next_18h`
  (measures "spike then die" shape; high value ⇒ meme-like)

Sustained news events have larger lifespans and lower peak-to-decay ratios,
while memes spike fast and flatline — these shape-of-the-curve signals let
the classifier separate them even when text looks similar.


In [ ]:
def _cluster_velocity_stats(df: pd.DataFrame) -> dict[int, dict]:
    """Compute per-cluster velocity features keyed by topic id."""
    stats: dict[int, dict] = {}
    for tid, sub in df.groupby("topic"):
        times = sub[TIME_COL].sort_values()
        size = len(times)
        first = times.iloc[0]
        last = times.iloc[-1]
        span_hours = (last - first).total_seconds() / 3600.0

        # Guard against zero-span clusters (all tweets at the same instant).
        safe_span = max(span_hours, 1e-3)
        rate = size / safe_span

        # peak-to-decay: tweets in first 6h vs next 18h (anchored at cluster start).
        early_cutoff = first + pd.Timedelta(hours=6)
        late_cutoff  = first + pd.Timedelta(hours=24)
        early = int(((times >= first) & (times < early_cutoff)).sum())
        late  = int(((times >= early_cutoff) & (times < late_cutoff)).sum())
        peak_to_decay = early / max(late, 1)  # avoid div-by-zero; high ⇒ spike-then-die

        stats[int(tid)] = {
            "tweet_rate_per_hour": float(rate),
            "lifespan_hours": float(span_hours),
            "peak_to_decay_ratio": float(peak_to_decay),
        }
    return stats


velocity_stats = _cluster_velocity_stats(labeled_df)

# Outliers (topic == -1) get neutral values: no rate, no lifespan, flat ratio.
DEFAULT_VEL = {
    "tweet_rate_per_hour": 0.0,
    "lifespan_hours": 0.0,
    "peak_to_decay_ratio": 1.0,
}


def _velocity_row(topic_id: int) -> np.ndarray:
    s = velocity_stats.get(int(topic_id), DEFAULT_VEL)
    return np.array(
        [s["tweet_rate_per_hour"], s["lifespan_hours"], s["peak_to_decay_ratio"]],
        dtype=np.float32,
    )


velocity_matrix = np.vstack([_velocity_row(t) for t in labeled_df["topic"].values])
features = np.hstack([embeddings, velocity_matrix]).astype(np.float32)
labels = labeled_df["label"].values.astype(np.int64)

np.save(FEATURES_NPY, features)
np.save(LABELS_NPY, labels)

print(f"features shape: {features.shape}   (embedding 768 + velocity 3)")
print(f"labels   shape: {labels.shape}     class counts: {np.bincount(labels)}")
print("\nVelocity stats per cluster:")
for tid, s in sorted(velocity_stats.items()):
    print(
        f"  topic {tid:>2}  rate/h={s['tweet_rate_per_hour']:.2f}"
        f"  lifespan={s['lifespan_hours']:.2f}h"
        f"  peak/decay={s['peak_to_decay_ratio']:.2f}"
    )


## 8. XGBoost Training

80/20 stratified split, trained with early stopping on the validation set.
Prints the classification report and the confusion matrix.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

# Stratify when both classes are actually present; otherwise fall back to random split.
stratify = labels if len(np.unique(labels)) > 1 else None

X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size=0.2,
    random_state=SEED,
    stratify=stratify,
)

# Balance classes via scale_pos_weight = neg / pos (protects against skew).
pos = max(int((y_train == 1).sum()), 1)
neg = max(int((y_train == 0).sum()), 1)
scale_pos_weight = neg / pos

clf = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)

clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
clf.save_model(XGB_MODEL_JSON)

y_pred = clf.predict(X_test)
print("Classification report:\n")
print(classification_report(y_test, y_pred, target_names=["meme", "real_event"], zero_division=0))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred))
print(f"\nSaved XGBoost model -> {XGB_MODEL_JSON}")


## 9. Inference

`classify_tweet(text)` — takes a raw tweet string and returns:

```python
{"label": "meme" | "real_event", "confidence": float, "probabilities": {...}}
```

Workflow:

1. clean text with the same preprocessor used in training
2. SBERT encode (with normalization so it matches training embeddings)
3. assign nearest known cluster via BERTopic's `transform` and pull its
   velocity features (fall back to neutral defaults if it hits the outlier topic)
4. concatenate and run the XGBoost classifier


In [ ]:
LABEL_NAMES = {0: "meme", 1: "real_event"}


def classify_tweet(text: str) -> dict:
    """End-to-end inference for a single raw tweet."""
    cleaned = clean_tweet(text)
    if not cleaned:
        return {"label": "meme", "confidence": 0.0, "probabilities": {"meme": 1.0, "real_event": 0.0}}

    emb = sbert.encode([cleaned], convert_to_numpy=True, normalize_embeddings=True)

    # BERTopic.transform returns the closest known topic (may be -1 = outlier).
    topic_id, _ = topic_model.transform([cleaned], embeddings=emb)
    topic_id = int(topic_id[0])
    velocity = _velocity_row(topic_id).reshape(1, -1)

    feat = np.hstack([emb, velocity]).astype(np.float32)
    proba = clf.predict_proba(feat)[0]
    pred = int(np.argmax(proba))

    return {
        "label": LABEL_NAMES[pred],
        "confidence": float(proba[pred]),
        "probabilities": {
            "meme": float(proba[0]),
            "real_event": float(proba[1]) if len(proba) > 1 else 0.0,
        },
        "matched_topic": topic_id,
    }


# Smoke test on a handful of inputs.
for sample in [
    "Massive 6.5 earthquake just rocked Istanbul, buildings swaying",
    "skibidi toilet ohio rizz level 9000 fr fr 💀",
    "AWS us-east-1 is throwing 500s across the board, APIs degraded",
]:
    print(sample)
    print("  ->", classify_tweet(sample))
    print()
